In [1]:
import pandas as pd

In [8]:
# Load datasets
listings = pd.read_csv("listings_with_rates.csv", low_memory=False)
sold = pd.read_csv("sold_with_rates.csv", low_memory=False)

In [9]:
def clean_dataset(df, name="dataset"):
    print(f"\n--- Cleaning {name} ---")
    
    before_rows = df.shape[0]

    # Convert date columns
    date_cols = [
        "CloseDate",
        "PurchaseContractDate",
        "ListingContractDate",
        "ContractStatusChangeDate"
    ]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # Remove unnecessary columns
    drop_cols = [col for col in df.columns if "Unnamed" in col]
    df = df.drop(columns=drop_cols, errors='ignore')

    # Ensure numeric types
    numeric_cols = [
        "ClosePrice", "LivingArea", "DaysOnMarket",
        "BedroomsTotal", "BathroomsTotalInteger", "Latitude", "Longitude"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Handle missing values
    df = df.dropna(subset=["ClosePrice", "LivingArea"], how='any')

    # Flag invalid numeric values
    df["invalid_price_flag"] = df["ClosePrice"] <= 0
    df["invalid_area_flag"] = df["LivingArea"] <= 0
    df["invalid_dom_flag"] = df["DaysOnMarket"] < 0
    df["invalid_beds_flag"] = df["BedroomsTotal"] < 0
    df["invalid_baths_flag"] = df["BathroomsTotalInteger"] < 0

    # Date consistency checks
    df["listing_after_close_flag"] = (
        df["ListingContractDate"] > df["CloseDate"]
    )
    df["purchase_after_close_flag"] = (
        df["PurchaseContractDate"] > df["CloseDate"]
    )
    df["negative_timeline_flag"] = (
        df["ListingContractDate"] > df["PurchaseContractDate"]
    )

    # Geographic checks
    df["missing_coords_flag"] = df["Latitude"].isna() | df["Longitude"].isna()
    df["zero_coords_flag"] = (df["Latitude"] == 0) | (df["Longitude"] == 0)
    df["positive_longitude_flag"] = df["Longitude"] > 0

    # California rough bounds 
    df["out_of_bounds_flag"] = (
        (df["Latitude"] < 32) | (df["Latitude"] > 42) |
        (df["Longitude"] < -125) | (df["Longitude"] > -114)
    )

    # Summary stats
    after_rows = df.shape[0]

    print(f"Rows before: {before_rows}")
    print(f"Rows after: {after_rows}")
    print("\nData types:\n", df.dtypes)

    print("\nDate flag counts:")
    print("listing_after_close:", df["listing_after_close_flag"].sum())
    print("purchase_after_close:", df["purchase_after_close_flag"].sum())
    print("negative_timeline:", df["negative_timeline_flag"].sum())

    print("\nGeographic flag counts:")
    print("missing_coords:", df["missing_coords_flag"].sum())
    print("zero_coords:", df["zero_coords_flag"].sum())
    print("positive_longitude:", df["positive_longitude_flag"].sum())
    print("out_of_bounds:", df["out_of_bounds_flag"].sum())

    return df

ERROR! Session/line number was not unique in database. History logging moved to new session 16


In [10]:
# Apply cleaning
listings_clean = clean_dataset(listings, "Listings")
sold_clean = clean_dataset(sold, "Sold")


--- Cleaning Listings ---
Rows before: 19630
Rows after: 8224

Data types:
 OriginalListPrice                 float64
ListingKey                          int64
CloseDate                  datetime64[ns]
ClosePrice                        float64
ListAgentFirstName                 object
                                ...      
negative_timeline_flag               bool
missing_coords_flag                  bool
zero_coords_flag                     bool
positive_longitude_flag              bool
out_of_bounds_flag                   bool
Length: 82, dtype: object

Date flag counts:
listing_after_close: 2
purchase_after_close: 0
negative_timeline: 12

Geographic flag counts:
missing_coords: 67
zero_coords: 1
positive_longitude: 0
out_of_bounds: 1

--- Cleaning Sold ---
Rows before: 749543
Rows after: 694719

Data types:
 Flooring                   object
ViewYN                     object
WaterfrontYN               object
BasementYN                 object
PoolPrivateYN              object
   

In [11]:
# Save outputs
listings_clean.to_csv("listings_cleaned.csv", index=False)
sold_clean.to_csv("sold_cleaned.csv", index=False)